In [2]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.6 MB/s  0:00:02 eta 0:00:010m
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install -q openai pandas openpyxl sentence-transformers scikit-learn

In [6]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from getpass import getpass
from openai import OpenAI

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# 1) PATHS
# =========================================================
data_dir = "/Users/shinekhantaung/Desktop/GPT-NER/5fold_train_test"
output_dir = os.path.join(
    "/Users/shinekhantaung/Desktop/GPT-NER/Sentence-Retrieval/sentence_similarity_results"
)
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# 2) API KEY (secure input)
# =========================================================
api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

# =========================================================
# 3) SETTINGS
# =========================================================
MODEL_NAME = "gpt-4.1"

# column names: dataset နဲ့မကိုက်ရင် ပြင်ပါ
sentence_col = "sentence"
name_col = "gold_names"

# paper style k values
k_values = [4, 8, 16, 32]

# sentence embedding model
# CPU အတွက် ပေါ့ပေါ့ပါးပါး baseline
# SIM_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# true SimCSE model သုံးချင်ရင် ဒီလိုပြောင်းနိုင်
SIM_MODEL_NAME = "princeton-nlp/sup-simcse-bert-base-uncased"
# SIM_MODEL_NAME = "sentence-transformers/sup-simcse-bert-base-uncased"

# retrieval details excel ထဲမှာ ဘယ်နှစ်ခုထိ save မလဲ
MAX_SAVE_DEMOS = 8

# rate limit soft protection
SLEEP_SECONDS = 0.2

# =========================================================
# 4) LOAD SENTENCE EMBEDDING MODEL
# =========================================================
print(f"Loading sentence embedding model: {SIM_MODEL_NAME}")
sim_model = SentenceTransformer(SIM_MODEL_NAME, device="cpu")
print("Sentence embedding model loaded.")

# =========================================================
# 5) HELPERS
# =========================================================
def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def normalize_entity_for_eval(s: str) -> str:
    """
    evaluation compare အတွက်ပဲ normalize လုပ်မယ်
    saved output / displayed prediction အတွက် မသုံးဘူး
    """
    s = normalize_text(s)
    return s.casefold()

def parse_gold_names(cell_value):
    """
    Gold name column က
    - blank / NaN
    - one name
    - multiple names separated by ; , |
    ဆိုတာတွေ handle လုပ်မယ်
    """
    if pd.isna(cell_value):
        return []

    text = str(cell_value).strip()
    if not text:
        return []

    parts = re.split(r"[;|,]", text)
    names = [normalize_text(x) for x in parts if normalize_text(x)]

    # deduplicate for display/output (original form ကိုပဲထား)
    seen = set()
    result = []
    for n in names:
        key = normalize_entity_for_eval(n)
        if key not in seen:
            seen.add(key)
            result.append(n)
    return result

def extract_marked_names(output_text: str):
    """
    @@name## pattern တွေ extract
    predicted names ကို normalize မလုပ်ဘူး
    sentence/output ထဲက original form အတိုင်းပဲထုတ်မယ်
    """
    if not isinstance(output_text, str):
        return []

    matches = re.findall(r"@@(.*?)##", output_text, flags=re.DOTALL)
    names = [normalize_text(m) for m in matches if normalize_text(m)]

    # deduplicate only, but preserve original extracted form
    seen = set()
    result = []
    for n in names:
        key = normalize_entity_for_eval(n)
        if key not in seen:
            seen.add(key)
            result.append(n)
    return result

def mark_gold_names(sentence: str, gold_names):
    """
    gold names ကို @@ and ## နဲ့ mark လုပ်ပေးမယ်
    longest first replace လုပ်မယ်
    """
    sentence = normalize_text(sentence)

    if not gold_names:
        return sentence

    marked = sentence
    gold_names_sorted = sorted(gold_names, key=len, reverse=True)

    for name in gold_names_sorted:
        name = normalize_text(name)
        if not name:
            continue

        pattern = re.escape(name)
        marked = re.sub(pattern, f"@@{name}##", marked)

    return marked

def build_prompt(test_sentence: str, demo_examples):
    """
    GPT-NER paper style few-shot prompt
    """
    prompt_lines = [
        "I am an excellent linguist.",
        "",
        "The task is to label person entities in the given Old English sentence using @@ and ##.",
        "If no person entity is present, return the sentence unchanged.",
        "",
        "Below are some examples.",
        ""
    ]

    for ex in demo_examples:
        demo_sentence = normalize_text(ex[sentence_col])
        demo_gold_names = parse_gold_names(ex[name_col])
        demo_output = mark_gold_names(demo_sentence, demo_gold_names)

        prompt_lines.append(f"Input: {demo_sentence}")
        prompt_lines.append(f"Output: {demo_output}")
        prompt_lines.append("")

    prompt_lines.append(f"Input: {test_sentence}")
    prompt_lines.append("Output:")

    return "\n".join(prompt_lines)

def call_gpt(prompt: str):
    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
    )

    output_text = getattr(response, "output_text", None)
    if output_text is None:
        try:
            output_text = response.output[0].content[0].text
        except Exception:
            output_text = str(response)

    return output_text.strip()

def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def load_train_test_fold(fold_id):
    train_path = os.path.join(data_dir, f"train_fold_{fold_id}.xlsx")
    test_path = os.path.join(data_dir, f"test_fold_{fold_id}.xlsx")

    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Train file not found: {train_path}")
    if not os.path.exists(test_path):
        raise FileNotFoundError(f"Test file not found: {test_path}")

    train_df = pd.read_excel(train_path)
    test_df = pd.read_excel(test_path)

    return train_df, test_df

def encode_sentences(sentences, batch_size=32):
    """
    train sentence embeddings precompute
    normalize_embeddings=True လုပ်ထားလို့ cosine similarity တိုက်ရိုက်သုံးလို့ရ
    """
    sentences = [normalize_text(s) for s in sentences]
    embeddings = sim_model.encode(
        sentences,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return embeddings

def get_topk_similar_demo_examples(test_sentence, train_df, train_embeddings, k):
    """
    test sentence embedding ကို query အနေနဲ့သုံးပြီး
    cosine similarity နဲ့ top-k nearest train sentences ရွေးမယ်
    """
    if len(train_df) == 0:
        return [], [], []

    test_sentence = normalize_text(test_sentence)

    test_embedding = sim_model.encode(
        [test_sentence],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    sims = cosine_similarity(test_embedding, train_embeddings)[0]

    top_k = min(k, len(train_df))
    top_indices = np.argsort(-sims)[:top_k]

    demo_examples = train_df.iloc[top_indices].to_dict("records")
    demo_scores = sims[top_indices].tolist()
    demo_sentences = train_df.iloc[top_indices][sentence_col].tolist()

    return demo_examples, demo_scores, demo_sentences

def build_retrieval_columns(demo_sentences, demo_scores, max_save_demos=MAX_SAVE_DEMOS):
    """
    retrieved demo sentence / score တွေကို excel save ဖို့ columns ပြန်ဆောက်မယ်
    """
    row = {}
    for idx in range(max_save_demos):
        sent_key = f"retrieved_demo_{idx + 1}"
        score_key = f"retrieval_score_{idx + 1}"

        row[sent_key] = demo_sentences[idx] if idx < len(demo_sentences) else ""
        row[score_key] = demo_scores[idx] if idx < len(demo_scores) else ""

    return row

# =========================================================
# 6) RUN ALL K VALUES
# =========================================================
all_k_summary = []

for k in k_values:
    print("\n" + "=" * 60)
    print(f"Running Sentence Similarity Retrieval with k = {k}")
    print("=" * 60)

    k_output_dir = os.path.join(output_dir, f"k_{k}")
    os.makedirs(k_output_dir, exist_ok=True)

    fold_results = []
    all_tp, all_fp, all_fn = 0, 0, 0

    for fold_id in range(1, 6):
        try:
            train_df, test_df = load_train_test_fold(fold_id)
        except Exception as e:
            print(f"[Skip] Fold {fold_id}: {e}")
            continue

        # column name check
        if sentence_col not in train_df.columns or name_col not in train_df.columns:
            print(f"[Error] Train Fold {fold_id}: columns not found.")
            print("Available columns:", train_df.columns.tolist())
            continue

        if sentence_col not in test_df.columns or name_col not in test_df.columns:
            print(f"[Error] Test Fold {fold_id}: columns not found.")
            print("Available columns:", test_df.columns.tolist())
            continue

        # normalize sentence column
        train_df[sentence_col] = train_df[sentence_col].apply(normalize_text)
        test_df[sentence_col] = test_df[sentence_col].apply(normalize_text)

        # -----------------------------------------
        # Precompute train embeddings once per fold
        # -----------------------------------------
        print(f"\nEncoding train sentences for Fold {fold_id} ...")
        train_sentences = train_df[sentence_col].tolist()
        train_embeddings = encode_sentences(train_sentences, batch_size=32)

        tp, fp, fn = 0, 0, 0
        row_outputs = []

        print(f"\n========== Fold {fold_id} | k = {k} ==========")
        print(f"Rows: {len(test_df)}")

        for i, row in test_df.iterrows():
            sentence = normalize_text(row[sentence_col])
            gold_names = parse_gold_names(row[name_col])

            try:
                demo_examples, demo_scores, demo_sentences = get_topk_similar_demo_examples(
                    test_sentence=sentence,
                    train_df=train_df,
                    train_embeddings=train_embeddings,
                    k=k
                )

                prompt = build_prompt(sentence, demo_examples)
                pred_output = call_gpt(prompt)
                pred_names = extract_marked_names(pred_output)

            except Exception as e:
                pred_output = f"[ERROR] {e}"
                pred_names = []
                demo_scores = []
                demo_sentences = []

            # -------------------------------------------------
            # IMPORTANT:
            # output/save အတွက် pred_names ကို original form အတိုင်းထားမယ်
            # evaluation compare မှာပဲ normalize လုပ်မယ်
            # -------------------------------------------------
            gold_set = {normalize_entity_for_eval(x) for x in gold_names}
            pred_set = {normalize_entity_for_eval(x) for x in pred_names}

            row_tp = len(gold_set & pred_set)
            row_fp = len(pred_set - gold_set)
            row_fn = len(gold_set - pred_set)

            tp += row_tp
            fp += row_fp
            fn += row_fn

            row_dict = {
                "sentence": sentence,
                "gold_name_raw": row[name_col],
                "gold_names_parsed": "; ".join(gold_names),
                "model_output": pred_output,
                "pred_names_parsed": "; ".join(pred_names),  # original extracted form
                "row_tp": row_tp,
                "row_fp": row_fp,
                "row_fn": row_fn,
            }

            row_dict.update(
                build_retrieval_columns(
                    demo_sentences=demo_sentences,
                    demo_scores=demo_scores,
                    max_save_demos=MAX_SAVE_DEMOS
                )
            )

            row_outputs.append(row_dict)

            time.sleep(SLEEP_SECONDS)

        precision, recall, f1 = compute_metrics(tp, fp, fn)

        print(f"TP={tp}, FP={fp}, FN={fn}")
        print(f"Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

        fold_results.append({
            "k": k,
            "fold": fold_id,
            "rows": len(test_df),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

        all_tp += tp
        all_fp += fp
        all_fn += fn

        # save detailed predictions
        detail_df = pd.DataFrame(row_outputs)
        detail_out = os.path.join(k_output_dir, f"predictions_fold_{fold_id}.xlsx")
        detail_df.to_excel(detail_out, index=False)

    # =========================================================
    # 7) SUMMARY FOR CURRENT K
    # =========================================================
    summary_df = pd.DataFrame(fold_results)

    if not summary_df.empty:
        macro_precision = summary_df["precision"].mean()
        macro_recall = summary_df["recall"].mean()
        macro_f1 = summary_df["f1"].mean()

        micro_precision, micro_recall, micro_f1 = compute_metrics(all_tp, all_fp, all_fn)

        print("\n========== Fold-wise Results ==========")
        print(summary_df)

        print(f"\n========== Average Across 5 Folds | k = {k} ==========")
        print(f"Macro Precision: {macro_precision:.4f}")
        print(f"Macro Recall   : {macro_recall:.4f}")
        print(f"Macro F1       : {macro_f1:.4f}")

        print("\n========== Overall Micro Score ==========")
        print(f"TP={all_tp}, FP={all_fp}, FN={all_fn}")
        print(f"Micro Precision: {micro_precision:.4f}")
        print(f"Micro Recall   : {micro_recall:.4f}")
        print(f"Micro F1       : {micro_f1:.4f}")

        summary_out = os.path.join(k_output_dir, "fold_summary.xlsx")
        summary_df.to_excel(summary_out, index=False)

        final_txt = os.path.join(k_output_dir, "final_scores.txt")
        with open(final_txt, "w", encoding="utf-8") as f:
            f.write(f"Sentence Similarity Retrieval Results | k = {k}\n\n")
            f.write("Fold-wise Results\n")
            f.write(summary_df.to_string(index=False))
            f.write("\n\nAverage Across 5 Folds\n")
            f.write(f"Macro Precision: {macro_precision:.4f}\n")
            f.write(f"Macro Recall   : {macro_recall:.4f}\n")
            f.write(f"Macro F1       : {macro_f1:.4f}\n\n")
            f.write("Overall Micro Score\n")
            f.write(f"TP={all_tp}, FP={all_fp}, FN={all_fn}\n")
            f.write(f"Micro Precision: {micro_precision:.4f}\n")
            f.write(f"Micro Recall   : {micro_recall:.4f}\n")
            f.write(f"Micro F1       : {micro_f1:.4f}\n")

        all_k_summary.append({
            "k": k,
            "macro_precision": macro_precision,
            "macro_recall": macro_recall,
            "macro_f1": macro_f1,
            "micro_tp": all_tp,
            "micro_fp": all_fp,
            "micro_fn": all_fn,
            "micro_precision": micro_precision,
            "micro_recall": micro_recall,
            "micro_f1": micro_f1,
        })

# =========================================================
# 8) FINAL SUMMARY FOR ALL K
# =========================================================
all_k_summary_df = pd.DataFrame(all_k_summary)

if not all_k_summary_df.empty:
    print("\n" + "=" * 60)
    print("FINAL SUMMARY FOR ALL K")
    print("=" * 60)
    print(all_k_summary_df)

    all_k_summary_out = os.path.join(output_dir, "all_k_summary.xlsx")
    all_k_summary_df.to_excel(all_k_summary_out, index=False)

    all_k_txt = os.path.join(output_dir, "all_k_summary.txt")
    with open(all_k_txt, "w", encoding="utf-8") as f:
        f.write("Final Summary for All K\n\n")
        f.write(all_k_summary_df.to_string(index=False))

    print(f"\nSaved results to: {output_dir}")
else:
    print("No results were generated.")

Loading sentence embedding model: princeton-nlp/sup-simcse-bert-base-uncased


No sentence-transformers model found with name princeton-nlp/sup-simcse-bert-base-uncased. Creating a new one with mean pooling.


Sentence embedding model loaded.

Running Sentence Similarity Retrieval with k = 4

Encoding train sentences for Fold 1 ...


Batches: 100%|██████████| 24/24 [00:10<00:00,  2.19it/s]



========== Fold 1 | k = 4 ==========
Rows: 186
TP=248, FP=26, FN=24
Precision=0.9051, Recall=0.9118, F1=0.9084

Encoding train sentences for Fold 2 ...


...


========== Fold 3 | k = 32 ==========
Rows: 186
TP=97, FP=9, FN=134
Precision=0.9151, Recall=0.4199, F1=0.5757

Encoding train sentences for Fold 4 ...


Batches: 100%|██████████| 24/24 [00:10<00:00,  2.27it/s]



========== Fold 4 | k = 32 ==========
Rows: 186
TP=104, FP=32, FN=144
Precision=0.7647, Recall=0.4194, F1=0.5417

Encoding train sentences for Fold 5 ...


Batches: 100%|██████████| 24/24 [00:10<00:00,  2.30it/s]



========== Fold 5 | k = 32 ==========
Rows: 185
TP=110, FP=7, FN=140
Precision=0.9402, Recall=0.4400, F1=0.5995

========== Fold-wise Results ==========
    k  fold  rows   tp  fp   fn  precision    recall        f1
0  32     1   186  104   4  168   0.962963  0.382353  0.547368
1  32     2   186  111   7  137   0.940678  0.447581  0.606557
2  32     3   186   97   9  134   0.915094  0.419913  0.575668
3  32     4   186  104  32  144   0.764706  0.419355  0.541667
4  32     5   185  110   7  140   0.940171  0.440000  0.599455

========== Average Across 5 Folds | k = 32 ==========
Macro Precision: 0.9047
Macro Recall   : 0.4218
Macro F1       : 0.5741

========== Overall Micro Score ==========
TP=526, FP=59, FN=723
Micro Precision: 0.8991
Micro Recall   : 0.4211
Micro F1       : 0.5736

FINAL SUMMARY FOR ALL K
    k  macro_precision  macro_recall  macro_f1  micro_tp  micro_fp  micro_fn  \
0   4         0.903135      0.887396  0.895096      1109       119       140   
1   8         0.912

### We had limitted gpt call error on k=16,32.


In [1]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from getpass import getpass
from openai import OpenAI

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# 1) PATHS
# =========================================================
data_dir = "/Users/shinekhantaung/Desktop/GPT-NER/5fold_train_test"
output_dir = os.path.join(
    "/Users/shinekhantaung/Desktop/GPT-NER/Sentence-Retrieval/sentence_similarity_results"
)
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# 2) API KEY (secure input)
# =========================================================
api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

# =========================================================
# 3) SETTINGS
# =========================================================
MODEL_NAME = "gpt-4.1"

# column names: dataset နဲ့မကိုက်ရင် ပြင်ပါ
sentence_col = "sentence"
name_col = "gold_names"

# rerun only problematic k values
k_values = [16, 32]

# sentence embedding model
SIM_MODEL_NAME = "princeton-nlp/sup-simcse-bert-base-uncased"
# SIM_MODEL_NAME = "sentence-transformers/sup-simcse-bert-base-uncased"

# retrieval details excel ထဲမှာ ဘယ်နှစ်ခုထိ save မလဲ
MAX_SAVE_DEMOS = 8

# old setting (မူရင်း structure မပျက်အောင်ထား)
SLEEP_SECONDS = 0.2

# retry settings
MAX_RETRIES = 5
BASE_RETRY_WAIT = 5

# cooldown settings
COOLDOWN_EVERY = 10
COOLDOWN_SECONDS = 5

# =========================================================
# 4) LOAD SENTENCE EMBEDDING MODEL
# =========================================================
print(f"Loading sentence embedding model: {SIM_MODEL_NAME}")
sim_model = SentenceTransformer(SIM_MODEL_NAME, device="cpu")
print("Sentence embedding model loaded.")

# =========================================================
# 5) HELPERS
# =========================================================
def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def normalize_entity_for_eval(s: str) -> str:
    """
    evaluation compare အတွက်ပဲ normalize လုပ်မယ်
    saved output / displayed prediction အတွက် မသုံးဘူး
    """
    s = normalize_text(s)
    return s.casefold()

def parse_gold_names(cell_value):
    """
    Gold name column က
    - blank / NaN
    - one name
    - multiple names separated by ; , |
    ဆိုတာတွေ handle လုပ်မယ်
    """
    if pd.isna(cell_value):
        return []

    text = str(cell_value).strip()
    if not text:
        return []

    parts = re.split(r"[;|,]", text)
    names = [normalize_text(x) for x in parts if normalize_text(x)]

    seen = set()
    result = []
    for n in names:
        key = normalize_entity_for_eval(n)
        if key not in seen:
            seen.add(key)
            result.append(n)
    return result

def extract_marked_names(output_text: str):
    """
    @@name## pattern တွေ extract
    predicted names ကို normalize မလုပ်ဘူး
    sentence/output ထဲက original form အတိုင်းပဲထုတ်မယ်
    """
    if not isinstance(output_text, str):
        return []

    matches = re.findall(r"@@(.*?)##", output_text, flags=re.DOTALL)
    names = [normalize_text(m) for m in matches if normalize_text(m)]

    seen = set()
    result = []
    for n in names:
        key = normalize_entity_for_eval(n)
        if key not in seen:
            seen.add(key)
            result.append(n)
    return result

def mark_gold_names(sentence: str, gold_names):
    """
    gold names ကို @@ and ## နဲ့ mark လုပ်ပေးမယ်
    longest first replace လုပ်မယ်
    """
    sentence = normalize_text(sentence)

    if not gold_names:
        return sentence

    marked = sentence
    gold_names_sorted = sorted(gold_names, key=len, reverse=True)

    for name in gold_names_sorted:
        name = normalize_text(name)
        if not name:
            continue

        pattern = re.escape(name)
        marked = re.sub(pattern, f"@@{name}##", marked)

    return marked

def build_prompt(test_sentence: str, demo_examples):
    """
    GPT-NER paper style few-shot prompt
    """
    prompt_lines = [
        "I am an excellent linguist.",
        "",
        "The task is to label person entities in the given Old English sentence using @@ and ##.",
        "If no person entity is present, return the sentence unchanged.",
        "",
        "Below are some examples.",
        ""
    ]

    for ex in demo_examples:
        demo_sentence = normalize_text(ex[sentence_col])
        demo_gold_names = parse_gold_names(ex[name_col])
        demo_output = mark_gold_names(demo_sentence, demo_gold_names)

        prompt_lines.append(f"Input: {demo_sentence}")
        prompt_lines.append(f"Output: {demo_output}")
        prompt_lines.append("")

    prompt_lines.append(f"Input: {test_sentence}")
    prompt_lines.append("Output:")

    return "\n".join(prompt_lines)

def get_sleep_time_by_k(k):
    if k <= 8:
        return 1
    elif k == 16:
        return 2
    else:  # k == 32
        return 3

def call_gpt(prompt: str):
    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=prompt,
            )

            output_text = getattr(response, "output_text", None)
            if output_text is None:
                try:
                    output_text = response.output[0].content[0].text
                except Exception:
                    output_text = str(response)

            return output_text.strip()

        except Exception as e:
            error_text = str(e).lower()

            if "rate_limit" in error_text or "429" in error_text:
                wait_time = BASE_RETRY_WAIT + attempt * 3
                print(f"[Rate Limit] Retry {attempt + 1}/{MAX_RETRIES} in {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise e

    raise RuntimeError("Max retries exceeded for GPT call.")

def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def load_train_test_fold(fold_id):
    train_path = os.path.join(data_dir, f"train_fold_{fold_id}.xlsx")
    test_path = os.path.join(data_dir, f"test_fold_{fold_id}.xlsx")

    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Train file not found: {train_path}")
    if not os.path.exists(test_path):
        raise FileNotFoundError(f"Test file not found: {test_path}")

    train_df = pd.read_excel(train_path)
    test_df = pd.read_excel(test_path)

    return train_df, test_df

def encode_sentences(sentences, batch_size=32):
    """
    train sentence embeddings precompute
    normalize_embeddings=True လုပ်ထားလို့ cosine similarity တိုက်ရိုက်သုံးလို့ရ
    """
    sentences = [normalize_text(s) for s in sentences]
    embeddings = sim_model.encode(
        sentences,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return embeddings

def get_topk_similar_demo_examples(test_sentence, train_df, train_embeddings, k):
    """
    test sentence embedding ကို query အနေနဲ့သုံးပြီး
    cosine similarity နဲ့ top-k nearest train sentences ရွေးမယ်
    """
    if len(train_df) == 0:
        return [], [], []

    test_sentence = normalize_text(test_sentence)

    test_embedding = sim_model.encode(
        [test_sentence],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    sims = cosine_similarity(test_embedding, train_embeddings)[0]

    top_k = min(k, len(train_df))
    top_indices = np.argsort(-sims)[:top_k]

    demo_examples = train_df.iloc[top_indices].to_dict("records")
    demo_scores = sims[top_indices].tolist()
    demo_sentences = train_df.iloc[top_indices][sentence_col].tolist()

    return demo_examples, demo_scores, demo_sentences

def build_retrieval_columns(demo_sentences, demo_scores, max_save_demos=MAX_SAVE_DEMOS):
    """
    retrieved demo sentence / score တွေကို excel save ဖို့ columns ပြန်ဆောက်မယ်
    """
    row = {}
    for idx in range(max_save_demos):
        sent_key = f"retrieved_demo_{idx + 1}"
        score_key = f"retrieval_score_{idx + 1}"

        row[sent_key] = demo_sentences[idx] if idx < len(demo_sentences) else ""
        row[score_key] = demo_scores[idx] if idx < len(demo_scores) else ""

    return row

# =========================================================
# 6) RERUN ONLY K=16 AND K=32
#    existing outputs in k_16 / k_32 folders will be overwritten
# =========================================================
for k in k_values:
    print("\n" + "=" * 60)
    print(f"Running Sentence Similarity Retrieval with k = {k}")
    print("=" * 60)

    k_output_dir = os.path.join(output_dir, f"k_{k}")
    os.makedirs(k_output_dir, exist_ok=True)

    fold_results = []
    all_tp, all_fp, all_fn = 0, 0, 0

    for fold_id in range(1, 6):
        try:
            train_df, test_df = load_train_test_fold(fold_id)
        except Exception as e:
            print(f"[Skip] Fold {fold_id}: {e}")
            continue

        if sentence_col not in train_df.columns or name_col not in train_df.columns:
            print(f"[Error] Train Fold {fold_id}: columns not found.")
            print("Available columns:", train_df.columns.tolist())
            continue

        if sentence_col not in test_df.columns or name_col not in test_df.columns:
            print(f"[Error] Test Fold {fold_id}: columns not found.")
            print("Available columns:", test_df.columns.tolist())
            continue

        train_df[sentence_col] = train_df[sentence_col].apply(normalize_text)
        test_df[sentence_col] = test_df[sentence_col].apply(normalize_text)

        print(f"\nEncoding train sentences for Fold {fold_id} ...")
        train_sentences = train_df[sentence_col].tolist()
        train_embeddings = encode_sentences(train_sentences, batch_size=32)

        tp, fp, fn = 0, 0, 0
        row_outputs = []

        print(f"\n========== Fold {fold_id} | k = {k} ==========")
        print(f"Rows: {len(test_df)}")

        for i, row in test_df.iterrows():
            sentence = normalize_text(row[sentence_col])
            gold_names = parse_gold_names(row[name_col])

            try:
                demo_examples, demo_scores, demo_sentences = get_topk_similar_demo_examples(
                    test_sentence=sentence,
                    train_df=train_df,
                    train_embeddings=train_embeddings,
                    k=k
                )

                prompt = build_prompt(sentence, demo_examples)
                pred_output = call_gpt(prompt)
                pred_names = extract_marked_names(pred_output)

            except Exception as e:
                pred_output = f"[ERROR] {e}"
                pred_names = []
                demo_scores = []
                demo_sentences = []

            gold_set = {normalize_entity_for_eval(x) for x in gold_names}
            pred_set = {normalize_entity_for_eval(x) for x in pred_names}

            row_tp = len(gold_set & pred_set)
            row_fp = len(pred_set - gold_set)
            row_fn = len(gold_set - pred_set)

            tp += row_tp
            fp += row_fp
            fn += row_fn

            row_dict = {
                "sentence": sentence,
                "gold_name_raw": row[name_col],
                "gold_names_parsed": "; ".join(gold_names),
                "model_output": pred_output,
                "pred_names_parsed": "; ".join(pred_names),
                "row_tp": row_tp,
                "row_fp": row_fp,
                "row_fn": row_fn,
            }

            row_dict.update(
                build_retrieval_columns(
                    demo_sentences=demo_sentences,
                    demo_scores=demo_scores,
                    max_save_demos=MAX_SAVE_DEMOS
                )
            )

            row_outputs.append(row_dict)

            sleep_time = get_sleep_time_by_k(k)
            time.sleep(sleep_time)

            if (i + 1) % COOLDOWN_EVERY == 0:
                print(f"[Cooldown] Processed {i + 1} rows. Sleeping {COOLDOWN_SECONDS}s...")
                time.sleep(COOLDOWN_SECONDS)

        precision, recall, f1 = compute_metrics(tp, fp, fn)

        print(f"TP={tp}, FP={fp}, FN={fn}")
        print(f"Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

        fold_results.append({
            "k": k,
            "fold": fold_id,
            "rows": len(test_df),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

        all_tp += tp
        all_fp += fp
        all_fn += fn

        # overwrite existing detailed predictions
        detail_df = pd.DataFrame(row_outputs)
        detail_out = os.path.join(k_output_dir, f"predictions_fold_{fold_id}.xlsx")
        detail_df.to_excel(detail_out, index=False)

    # =========================================================
    # 7) SUMMARY FOR CURRENT K ONLY
    #    this will overwrite existing fold_summary.xlsx / final_scores.txt
    # =========================================================
    summary_df = pd.DataFrame(fold_results)

    if not summary_df.empty:
        macro_precision = summary_df["precision"].mean()
        macro_recall = summary_df["recall"].mean()
        macro_f1 = summary_df["f1"].mean()

        micro_precision, micro_recall, micro_f1 = compute_metrics(all_tp, all_fp, all_fn)

        print("\n========== Fold-wise Results ==========")
        print(summary_df)

        print(f"\n========== Average Across 5 Folds | k = {k} ==========")
        print(f"Macro Precision: {macro_precision:.4f}")
        print(f"Macro Recall   : {macro_recall:.4f}")
        print(f"Macro F1       : {macro_f1:.4f}")

        print("\n========== Overall Micro Score ==========")
        print(f"TP={all_tp}, FP={all_fp}, FN={all_fn}")
        print(f"Micro Precision: {micro_precision:.4f}")
        print(f"Micro Recall   : {micro_recall:.4f}")
        print(f"Micro F1       : {micro_f1:.4f}")

        summary_out = os.path.join(k_output_dir, "fold_summary.xlsx")
        summary_df.to_excel(summary_out, index=False)

        final_txt = os.path.join(k_output_dir, "final_scores.txt")
        with open(final_txt, "w", encoding="utf-8") as f:
            f.write(f"Sentence Similarity Retrieval Results | k = {k}\n\n")
            f.write("Fold-wise Results\n")
            f.write(summary_df.to_string(index=False))
            f.write("\n\nAverage Across 5 Folds\n")
            f.write(f"Macro Precision: {macro_precision:.4f}\n")
            f.write(f"Macro Recall   : {macro_recall:.4f}\n")
            f.write(f"Macro F1       : {macro_f1:.4f}\n\n")
            f.write("Overall Micro Score\n")
            f.write(f"TP={all_tp}, FP={all_fp}, FN={all_fn}\n")
            f.write(f"Micro Precision: {micro_precision:.4f}\n")
            f.write(f"Micro Recall   : {micro_recall:.4f}\n")
            f.write(f"Micro F1       : {micro_f1:.4f}\n")



/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading sentence embedding model: princeton-nlp/sup-simcse-bert-base-uncased


No sentence-transformers model found with name princeton-nlp/sup-simcse-bert-base-uncased. Creating a new one with mean pooling.


Sentence embedding model loaded.

Running Sentence Similarity Retrieval with k = 16

Encoding train sentences for Fold 1 ...


Batches: 100%|██████████| 24/24 [00:10<00:00,  2.22it/s]


...


========== Fold 3 | k = 32 ==========
Rows: 186
[Cooldown] Processed 10 rows. Sleeping 5s...
[Cooldown] Processed 20 rows. Sleeping 5s...
[Cooldown] Processed 30 rows. Sleeping 5s...
[Cooldown] Processed 40 rows. Sleeping 5s...
[Cooldown] Processed 50 rows. Sleeping 5s...
[Cooldown] Processed 60 rows. Sleeping 5s...
[Cooldown] Processed 70 rows. Sleeping 5s...
[Cooldown] Processed 80 rows. Sleeping 5s...
[Cooldown] Processed 90 rows. Sleeping 5s...
[Cooldown] Processed 100 rows. Sleeping 5s...
[Cooldown] Processed 110 rows. Sleeping 5s...
[Cooldown] Processed 120 rows. Sleeping 5s...
[Cooldown] Processed 130 rows. Sleeping 5s...
[Cooldown] Processed 140 rows. Sleeping 5s...
[Cooldown] Processed 150 rows. Sleeping 5s...
[Cooldown] Processed 160 rows. Sleeping 5s...
[Cooldown] Processed 170 rows. Sleeping 5s...
[Cooldown] Processed 180 rows. Sleeping 5s...
TP=217, FP=16, FN=14
Precision=0.9313, Recall=0.9394, F1=0.9353

Encoding train sentences for Fold 4 ...


Batches: 100%|██████████| 24/24 [00:10<00:00,  2.28it/s]



========== Fold 4 | k = 32 ==========
Rows: 186
[Cooldown] Processed 10 rows. Sleeping 5s...
[Cooldown] Processed 20 rows. Sleeping 5s...
[Cooldown] Processed 30 rows. Sleeping 5s...
[Cooldown] Processed 40 rows. Sleeping 5s...
[Cooldown] Processed 50 rows. Sleeping 5s...
[Cooldown] Processed 60 rows. Sleeping 5s...
[Cooldown] Processed 70 rows. Sleeping 5s...
[Cooldown] Processed 80 rows. Sleeping 5s...
[Cooldown] Processed 90 rows. Sleeping 5s...
[Cooldown] Processed 100 rows. Sleeping 5s...
[Cooldown] Processed 110 rows. Sleeping 5s...
[Cooldown] Processed 120 rows. Sleeping 5s...
[Cooldown] Processed 130 rows. Sleeping 5s...
[Cooldown] Processed 140 rows. Sleeping 5s...
[Cooldown] Processed 150 rows. Sleeping 5s...
[Cooldown] Processed 160 rows. Sleeping 5s...
[Cooldown] Processed 170 rows. Sleeping 5s...
[Cooldown] Processed 180 rows. Sleeping 5s...
TP=235, FP=42, FN=13
Precision=0.8484, Recall=0.9476, F1=0.8952

Encoding train sentences for Fold 5 ...


Batches: 100%|██████████| 24/24 [00:10<00:00,  2.39it/s]



========== Fold 5 | k = 32 ==========
Rows: 185
[Cooldown] Processed 10 rows. Sleeping 5s...
[Cooldown] Processed 20 rows. Sleeping 5s...
[Cooldown] Processed 30 rows. Sleeping 5s...
[Cooldown] Processed 40 rows. Sleeping 5s...
[Cooldown] Processed 50 rows. Sleeping 5s...
[Cooldown] Processed 60 rows. Sleeping 5s...
[Cooldown] Processed 70 rows. Sleeping 5s...
[Cooldown] Processed 80 rows. Sleeping 5s...
[Cooldown] Processed 90 rows. Sleeping 5s...
[Cooldown] Processed 100 rows. Sleeping 5s...
[Cooldown] Processed 110 rows. Sleeping 5s...
[Cooldown] Processed 120 rows. Sleeping 5s...
[Cooldown] Processed 130 rows. Sleeping 5s...
[Cooldown] Processed 140 rows. Sleeping 5s...
[Cooldown] Processed 150 rows. Sleeping 5s...
[Cooldown] Processed 160 rows. Sleeping 5s...
[Cooldown] Processed 170 rows. Sleeping 5s...
[Cooldown] Processed 180 rows. Sleeping 5s...
TP=239, FP=16, FN=11
Precision=0.9373, Recall=0.9560, F1=0.9465

========== Fold-wise Results ==========
    k  fold  rows   tp  fp 

In [3]:
import os
import re
import pandas as pd

# =========================================================
# PATH
# =========================================================
output_dir = "/Users/shinekhantaung/Desktop/GPT-NER/Sentence-Retrieval/sentence_similarity_results"

# =========================================================
# HELPERS
# =========================================================
def extract_metrics_from_txt(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()

    def extract_float(pattern):
        m = re.search(pattern, text)
        return float(m.group(1)) if m else None

    def extract_int(pattern):
        m = re.search(pattern, text)
        return int(m.group(1)) if m else None

    return {
        "macro_precision": extract_float(r"Macro Precision:\s*([0-9.]+)"),
        "macro_recall": extract_float(r"Macro Recall\s*:\s*([0-9.]+)"),
        "macro_f1": extract_float(r"Macro F1\s*:\s*([0-9.]+)"),
        "micro_tp": extract_int(r"TP=(\d+)"),
        "micro_fp": extract_int(r"FP=(\d+)"),
        "micro_fn": extract_int(r"FN=(\d+)"),
        "micro_precision": extract_float(r"Micro Precision:\s*([0-9.]+)"),
        "micro_recall": extract_float(r"Micro Recall\s*:\s*([0-9.]+)"),
        "micro_f1": extract_float(r"Micro F1\s*:\s*([0-9.]+)"),
    }

# =========================================================
# LOAD ALL K RESULTS
# =========================================================
k_values = [4, 8, 16, 32]
all_k_summary = []

for k in k_values:
    k_dir = os.path.join(output_dir, f"k_{k}")
    txt_path = os.path.join(k_dir, "final_scores.txt")

    if not os.path.exists(txt_path):
        print(f"[Skip] final_scores.txt not found for k={k}: {txt_path}")
        continue

    metrics = extract_metrics_from_txt(txt_path)
    metrics["k"] = k
    all_k_summary.append(metrics)

# =========================================================
# SAVE FINAL ALL-K SUMMARY
# =========================================================
all_k_summary_df = pd.DataFrame(all_k_summary)

if not all_k_summary_df.empty:
    all_k_summary_df = all_k_summary_df[
        [
            "k",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "micro_tp",
            "micro_fp",
            "micro_fn",
            "micro_precision",
            "micro_recall",
            "micro_f1",
        ]
    ].sort_values("k").reset_index(drop=True)

    print("\n" + "=" * 60)
    print("FINAL SUMMARY FOR ALL K")
    print("=" * 60)
    print(all_k_summary_df)

    all_k_summary_out = os.path.join(output_dir, "all_k_summary.xlsx")
    all_k_summary_df.to_excel(all_k_summary_out, index=False)

    all_k_txt = os.path.join(output_dir, "all_k_summary.txt")
    with open(all_k_txt, "w", encoding="utf-8") as f:
        f.write("Final Summary for All K\n\n")
        f.write(all_k_summary_df.to_string(index=False))

    print(f"\nSaved all-k summary to: {output_dir}")
else:
    print("No results were found.")


FINAL SUMMARY FOR ALL K
    k  macro_precision  macro_recall  macro_f1  micro_tp  micro_fp  micro_fn  \
0   4           0.9031        0.8874    0.8951      1109       119       140   
1   8           0.9121        0.9321    0.9219      1164       112        85   
2  16           0.9110        0.9424    0.9262      1177       116        72   
3  32           0.9142        0.9485    0.9307      1185       113        64   

   micro_precision  micro_recall  micro_f1  
0           0.9031        0.8879    0.8954  
1           0.9122        0.9319    0.9220  
2           0.9103        0.9424    0.9260  
3           0.9129        0.9488    0.9305  

Saved all-k summary to: /Users/shinekhantaung/Desktop/GPT-NER/Sentence-Retrieval/sentence_similarity_results
